In [ ]:
# Paths in this notebook are relative to the repository root; move there when opened from analysis/.
import os
from pathlib import Path
if not Path('Experiments_constraints').is_dir() and (Path('..') / 'Experiments_constraints').is_dir():
    os.chdir('..')
print('CWD:', os.getcwd())

#### test tsai wu

In [16]:
from joblib import dump, load
import numpy as np

tsai_wu = load('surrogate_tsaiwu.joblib')

In [17]:
import torch

def sample_sobol_on_grid(n=1, device=None, seed=123): # 21*21 = 441
    device = device or torch.device("cpu")
    D = 18
    var_dims = [0, 17]  # variable dimensions
    V = len(var_dims)

    # Sobol in [0,1]^V
    sobol = torch.quasirandom.SobolEngine(dimension=V, scramble=True, seed=seed)
    u = sobol.draw(n).to(device)

    # Map to continuous range [-1, 1]
    x = 2.0 * u - 1.0  # u in [0,1] -> [-1,1]

    # Snap to nearest grid in {-1.0, -0.9, ..., 1.0}
    values = torch.linspace(-1, 1, 21, device=device)           # 21 points
    step = values[1] - values[0]                                    # 0.1
    idx = torch.round((x - values[0]) / step).clamp(0, values.numel()-1).long()

    out = torch.zeros(n, D, device=device)
    out[:, var_dims] = values[idx]
    return out

search_space = sample_sobol_on_grid(n=441)
import scipy.stats as stats
X_np = search_space.detach().cpu().numpy()
mu_np, std_np = tsai_wu.predict(X_np, return_std=True)
std_np = np.maximum(std_np, 1e-12)
z_np = (1 - mu_np) / std_np
pof_np = stats.norm.cdf(z_np)

In [18]:
np.sum(mu_np - 1 >0)

132

In [19]:
import torch
torch.linspace(-1, 1, 41)

tensor([-1.0000e+00, -9.5000e-01, -9.0000e-01, -8.5000e-01, -8.0000e-01,
        -7.5000e-01, -7.0000e-01, -6.5000e-01, -6.0000e-01, -5.5000e-01,
        -5.0000e-01, -4.5000e-01, -4.0000e-01, -3.5000e-01, -3.0000e-01,
        -2.5000e-01, -2.0000e-01, -1.5000e-01, -1.0000e-01, -5.0000e-02,
        -1.4901e-08,  5.0000e-02,  1.0000e-01,  1.5000e-01,  2.0000e-01,
         2.5000e-01,  3.0000e-01,  3.5000e-01,  4.0000e-01,  4.5000e-01,
         5.0000e-01,  5.5000e-01,  6.0000e-01,  6.5000e-01,  7.0000e-01,
         7.5000e-01,  8.0000e-01,  8.5000e-01,  9.0000e-01,  9.5000e-01,
         1.0000e+00])

In [20]:
torch.randint(0, 1681, (3,))

tensor([ 257, 1087, 1602])

In [21]:
random_index = torch.randint(0, search_space.shape[0], (10,))
actions = search_space[random_index]

In [23]:
actions.shape

torch.Size([10, 18])